This notebook compares vegetation model v1.0.5 chunk stats against flox zonal stats by variable, year, and tile_id.
The two outputs should match.
I used a notebook because both outputs are so large that they cannot be opened in Excel and there are lots of combinations (356 tiles x 9 year x >20 outputs) to compare.
I did this with Claude session 'Flux statistics comparison: chunk vs. flox'.

Results:
Chunk stats and flox zonal stats matched for almost all tiles but didn't match for a sizable number in Canada. 
I investigated this and the fault is ultimately on the chunk stats side.
When I ran v1.0.5 on 2026-01-30, chunk stats didn't sum a variable if there was a NaN in the chunk.
(I corrected this later, in time for SOC v1.0.1 run.)
So, chunk stats was just dropping emissions for chunks that had NaN in them,
which was happening specifically in Canada for AGC and BGC emissions but not deadwood and litter. 
The reason NaN was happening for emissions was because I was missing an emission factor in my partial emission factor table.
The emission factor was missing only was for wildfire driver (code 5) in ecozone 2003 (N. American Boreal tundra woodland).
So, this happened under very limited circumstances and affects a pretty restricted set of pixels.

In [37]:
import pandas as pd
import numpy as np
import yaml
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import seaborn as sns
import re
import glob
import sys
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

In [64]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

In [3]:
# Vegetation zonal stats output
zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run__20260601/'

In [9]:
# Vegetation chunk stats output
chunk_stats_folder = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/parquet_20260131_10_37_46__KEEP/'
gross_outputs_parquet = f'{chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__gross_outputs_1x1.parquet'
net_outputs_parquet = f'{chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__net_outputs_1x1.parquet'

# Compares chunk stats and zonal stats sums for tile x year x variable combinations
### Per Claude session 'Flux statistics comparison: chunk vs. flox'

In [12]:
chunk_stats_gross = pd.read_parquet(gross_outputs_parquet)
chunk_stats_gross

,chunk_id,tile_id,layer_name,pattern,years,chunk_name,tile_name,in_out,min_value,mean_value,max_value,count_value,sum_value,data_type,iso
0,-28_-60_-27_-59,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2016,gross_emissions__AGC__MgCO2_ha_yr,2016,50S_030W__-28_-60_-27_-59__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
1,-27_-60_-26_-59,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2016,gross_emissions__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-60_-26_-59__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
2,177_-60_178_-59,50S_170E,gross_emissions__AGC__MgCO2_ha_yr_2016,gross_emissions__AGC__MgCO2_ha_yr,2016,50S_170E__177_-60_178_-59__gross_emissions__AG...,50S_170E__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,ATA
3,-27_-59_-26_-58,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2016,gross_emissions__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-59_-26_-58__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
4,-27_-58_-26_-57,50S_030W,gross_emissions__AGC__MgCO2_ha_yr_2016,gross_emissions__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-58_-26_-57__gross_emissions__AG...,50S_030W__gross_emissions__AGC__MgCO2_ha_yr_20...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2372827,100_79_101_80,80N_100E,gross_removals__litter_C__MgCO2_ha_yr_2024,gross_removals__litter_C__MgCO2_ha_yr,2024,80N_100E__100_79_101_80__gross_removals__litte...,80N_100E__gross_removals__litter_C__MgCO2_ha_y...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
2372828,101_79_102_80,80N_100E,gross_removals__litter_C__MgCO2_ha_yr_2024,gross_removals__litter_C__MgCO2_ha_yr,2024,80N_100E__101_79_102_80__gross_removals__litte...,80N_100E__gross_removals__litter_C__MgCO2_ha_y...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
2372829,102_79_103_80,80N_100E,gross_removals__litter_C__MgCO2_ha_yr_2024,gross_removals__litter_C__MgCO2_ha_yr,2024,80N_100E__102_79_103_80__gross_removals__litte...,80N_100E__gross_removals__litter_C__MgCO2_ha_y...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
2372830,103_79_104_80,80N_100E,gross_removals__litter_C__MgCO2_ha_yr_2024,gross_removals__litter_C__MgCO2_ha_yr,2024,80N_100E__103_79_104_80__gross_removals__litte...,80N_100E__gross_removals__litter_C__MgCO2_ha_y...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS


In [13]:
chunk_stats_net = pd.read_parquet(net_outputs_parquet)
chunk_stats_net

,chunk_id,tile_id,layer_name,pattern,years,chunk_name,tile_name,in_out,min_value,mean_value,max_value,count_value,sum_value,data_type,iso
0,-28_-60_-27_-59,50S_030W,net_flux__AGC__MgCO2_ha_yr_2016,net_flux__AGC__MgCO2_ha_yr,2016,50S_030W__-28_-60_-27_-59__net_flux__AGC__MgCO...,50S_030W__net_flux__AGC__MgCO2_ha_yr_2016.tif,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
1,-27_-60_-26_-59,50S_030W,net_flux__AGC__MgCO2_ha_yr_2016,net_flux__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-60_-26_-59__net_flux__AGC__MgCO...,50S_030W__net_flux__AGC__MgCO2_ha_yr_2016.tif,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
2,177_-60_178_-59,50S_170E,net_flux__AGC__MgCO2_ha_yr_2016,net_flux__AGC__MgCO2_ha_yr,2016,50S_170E__177_-60_178_-59__net_flux__AGC__MgCO...,50S_170E__net_flux__AGC__MgCO2_ha_yr_2016.tif,output_layer,NaN,NaN,NaN,NaN,0.0,no data,ATA
3,-27_-59_-26_-58,50S_030W,net_flux__AGC__MgCO2_ha_yr_2016,net_flux__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-59_-26_-58__net_flux__AGC__MgCO...,50S_030W__net_flux__AGC__MgCO2_ha_yr_2016.tif,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
4,-27_-58_-26_-57,50S_030W,net_flux__AGC__MgCO2_ha_yr_2016,net_flux__AGC__MgCO2_ha_yr,2016,50S_030W__-27_-58_-26_-57__net_flux__AGC__MgCO...,50S_030W__net_flux__AGC__MgCO2_ha_yr_2016.tif,output_layer,NaN,NaN,NaN,NaN,0.0,no data,SGS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1016923,100_79_101_80,80N_100E,net_flux__litter_C__MgCO2_ha_yr_2024,net_flux__litter_C__MgCO2_ha_yr,2024,80N_100E__100_79_101_80__net_flux__litter_C__M...,80N_100E__net_flux__litter_C__MgCO2_ha_yr_2024...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
1016924,101_79_102_80,80N_100E,net_flux__litter_C__MgCO2_ha_yr_2024,net_flux__litter_C__MgCO2_ha_yr,2024,80N_100E__101_79_102_80__net_flux__litter_C__M...,80N_100E__net_flux__litter_C__MgCO2_ha_yr_2024...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
1016925,102_79_103_80,80N_100E,net_flux__litter_C__MgCO2_ha_yr_2024,net_flux__litter_C__MgCO2_ha_yr,2024,80N_100E__102_79_103_80__net_flux__litter_C__M...,80N_100E__net_flux__litter_C__MgCO2_ha_yr_2024...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS
1016926,103_79_104_80,80N_100E,net_flux__litter_C__MgCO2_ha_yr_2024,net_flux__litter_C__MgCO2_ha_yr,2024,80N_100E__103_79_104_80__net_flux__litter_C__M...,80N_100E__net_flux__litter_C__MgCO2_ha_yr_2024...,output_layer,NaN,NaN,NaN,NaN,0.0,no data,RUS


In [47]:
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
chunk_stats_combined = pd.concat([chunk_stats_gross, chunk_stats_net], ignore_index=True)

chunk_tile_agg = (
    chunk_stats_combined
    .groupby(['tile_id', 'pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
chunk_tile_agg['year'] = chunk_tile_agg['year'].astype(int)
chunk_tile_agg['variable'] = chunk_tile_agg['variable'].str.replace('_ha_yr', '', regex=False)  # Removes _ha_yr to match zonal stats output
chunk_tile_agg

,tile_id,variable,year,tile_sum
0,00N_000E,gross_emissions__AGC__MgCO2,2016,6.803591e+05
1,00N_000E,gross_emissions__AGC__MgCO2,2017,1.055552e+06
2,00N_000E,gross_emissions__AGC__MgCO2,2018,8.115753e+05
3,00N_000E,gross_emissions__AGC__MgCO2,2019,9.394097e+05
4,00N_000E,gross_emissions__AGC__MgCO2,2020,1.174674e+06
...,...,...,...,...
64075,80N_180W,net_flux__litter_C__MgCO2,2020,0.000000e+00
64076,80N_180W,net_flux__litter_C__MgCO2,2021,0.000000e+00
64077,80N_180W,net_flux__litter_C__MgCO2,2022,0.000000e+00
64078,80N_180W,net_flux__litter_C__MgCO2,2023,0.000000e+00


In [59]:
# Iterates through flox zonal stats parquets to consolidate them to key variables and outputs

flox_parquet_files = sorted(glob.glob(f'{zonal_stats_folder}*.parquet'))
print(f"Found {len(flox_parquet_files)} flox parquet files")

tile_aggs = []
for f in tqdm(flox_parquet_files):
# for f in tqdm(flox_parquet_files[0:2]):  # To test a subset of tiles
    # print(f"Grouping {f}")
    df_tile = pd.read_parquet(f)
    agg = (
        df_tile
        .groupby(['tile_id', 'analysis_layer', 'year'], dropna=False)['value']
        .sum()
        .reset_index()
    )
    tile_aggs.append(agg)

# Combines simplified tile dfs, and drops EF and RF because those aren't in chunk stats output
flox_tile_agg = (
    pd.concat(tile_aggs, ignore_index=True)
    .pipe(lambda df: df[~df['analysis_layer'].isin([
        'AGC_emission_factor_CO2_only__fraction',
        'removal_factor__AGC__MgC',
        'carbon_density__non_soil__MgC_ha',
    ])])
    .rename(columns={'analysis_layer': 'variable', 'value': 'flox_sum'})
)
flox_tile_agg['year'] = flox_tile_agg['year'].astype(int)
flox_tile_agg

Found 356 flox parquet files


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 356/356 [03:25<00:00,  1.74it/s]


,tile_id,variable,year,flox_sum
18,00N_000E,gross_emissions__AGC__MgCO2,2016,6.803592e+05
19,00N_000E,gross_emissions__AGC__MgCO2,2017,1.055553e+06
20,00N_000E,gross_emissions__AGC__MgCO2,2018,8.115752e+05
21,00N_000E,gross_emissions__AGC__MgCO2,2019,9.394093e+05
22,00N_000E,gross_emissions__AGC__MgCO2,2020,1.174673e+06
...,...,...,...,...
55632,80N_170W,net_flux__litter_C__MgCO2,2020,-1.961436e-02
55633,80N_170W,net_flux__litter_C__MgCO2,2021,-2.119017e-02
55634,80N_170W,net_flux__litter_C__MgCO2,2022,-1.517800e-03
55635,80N_170W,net_flux__litter_C__MgCO2,2023,-2.300652e-02


In [60]:
# Merges chunk stats and zonal stats output tables to get a combined table for comparison

comparison = pd.merge(
    chunk_tile_agg,
    flox_tile_agg,
    on=['tile_id', 'variable', 'year'],
    how='outer',
    indicator=True
)
comparison

only_in_chunk = comparison[comparison['_merge'] == 'left_only']
only_in_flox  = comparison[comparison['_merge'] == 'right_only']
print(f"Tile×variable×year only in chunk stats: {len(only_in_chunk)}")
print(f"Tile×variable×year only in flox stats:  {len(only_in_flox)}")

comparison = comparison[comparison['_merge'] == 'both'].drop(columns='_merge')
comparison

comparison['abs_diff'] = comparison['flox_sum'] - comparison['tile_sum']
comparison['pct_diff'] = np.where(
    comparison['tile_sum'] == 0,
    np.where(comparison['flox_sum'] == 0, 0.0, np.inf),
    (comparison['flox_sum'] - comparison['tile_sum']) / comparison['tile_sum'].abs() * 100
)
comparison

Tile×variable×year only in chunk stats: 16201
Tile×variable×year only in flox stats:  0


,tile_id,variable,year,tile_sum,flox_sum,abs_diff,pct_diff
0,00N_000E,gross_emissions__AGC__MgCO2,2016,6.803591e+05,6.803592e+05,6.347656e-02,0.000009
1,00N_000E,gross_emissions__AGC__MgCO2,2017,1.055552e+06,1.055553e+06,7.070312e-01,0.000067
2,00N_000E,gross_emissions__AGC__MgCO2,2018,8.115753e+05,8.115752e+05,-9.252930e-02,-0.000011
3,00N_000E,gross_emissions__AGC__MgCO2,2019,9.394097e+05,9.394093e+05,-3.912354e-01,-0.000042
4,00N_000E,gross_emissions__AGC__MgCO2,2020,1.174674e+06,1.174673e+06,-4.370117e-01,-0.000037
...,...,...,...,...,...,...,...
63895,80N_170W,net_flux__litter_C__MgCO2,2020,-1.961436e-02,-1.961436e-02,0.000000e+00,0.000000
63896,80N_170W,net_flux__litter_C__MgCO2,2021,-2.119017e-02,-2.119017e-02,4.656613e-10,0.000002
63897,80N_170W,net_flux__litter_C__MgCO2,2022,-1.517800e-03,-1.517800e-03,1.164153e-10,0.000008
63898,80N_170W,net_flux__litter_C__MgCO2,2023,-2.300652e-02,-2.300652e-02,2.328306e-09,0.000010


In [62]:
# Reports all tile x year x variable combination sums that don't match between chunk stats and zonal stats

MISMATCH_THRESHOLD_PCT = 0.1  # 0.1% difference threshold

mismatches = (
    comparison[comparison['pct_diff'].abs() > MISMATCH_THRESHOLD_PCT]
    .sort_values('pct_diff', key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

print(f"Total tile×variable×year combinations compared: {len(comparison)}")
print(f"Mismatches (|pct_diff| > {MISMATCH_THRESHOLD_PCT}%):  {len(mismatches)}")
print(f"Matching:                                        {len(comparison) - len(mismatches)}")
display(mismatches)

tile_summary = (
    mismatches
    .groupby('tile_id')
    .agg(
        n_mismatches=('variable', 'count'),
        max_abs_pct_diff=('pct_diff', lambda x: x.abs().max()),
        variables=('variable', lambda x: sorted(x.unique()))
    )
    .sort_values('max_abs_pct_diff', ascending=False)
    .reset_index()
)
print(f"\nTiles with at least one mismatch: {len(tile_summary)}")
display(tile_summary)

Total tile×variable×year combinations compared: 47879
Mismatches (|pct_diff| > 0.1%):  1343
Matching:                                        46536


,tile_id,variable,year,tile_sum,flox_sum,abs_diff,pct_diff
0,70N_120W,gross_emissions__AGC__MgCO2,2017,2.230077e+04,1.661048e+07,1.658817e+07,74383.864978
1,70N_120W,gross_emissions__all_C_pools__all_gases__MgCO2e,2017,3.810222e+04,1.841743e+07,1.837933e+07,48236.891133
2,70N_120W,gross_emissions__all_C_pools__CO2_only__MgCO2,2017,3.740747e+04,1.745271e+07,1.741530e+07,46555.681967
3,70N_120W,gross_emissions__AGC__MgCO2,2023,1.593311e+05,6.954790e+07,6.938857e+07,43549.921424
4,70N_120W,gross_emissions__all_C_pools__all_gases__MgCO2e,2023,1.950542e+05,7.734099e+07,7.714594e+07,39551.030422
...,...,...,...,...,...,...,...
1338,70N_140W,net_flux__BGC__MgCO2,2024,-7.584358e+06,-7.601762e+06,-1.740420e+04,-0.229475
1339,70N_170W,gross_emissions__all_C_pools__all_gases__MgCO2e,2019,1.918142e+06,1.922541e+06,4.399789e+03,0.229378
1340,70N_170W,gross_emissions__AGC__MgCO2,2019,1.606618e+06,1.610026e+06,3.407983e+03,0.212122
1341,60N_110W,net_flux__all_C_pools__CO2_only__MgCO2,2023,7.271171e+07,7.257618e+07,-1.355224e+05,-0.186383



Tiles with at least one mismatch: 17


,tile_id,n_mismatches,max_abs_pct_diff,variables
0,70N_120W,93,74383.864978,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
1,70N_130W,87,23664.548592,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
2,60N_130W,87,11523.237173,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
3,70N_110W,81,7569.795615,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
4,60N_090W,87,2159.946740,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
5,70N_160W,90,2044.258930,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
6,60N_120W,93,1424.936798,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
7,70N_100W,24,998.677423,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
8,70N_150W,84,435.013042,"[gross_emissions__AGC__MgCO2, gross_emissions_..."
9,60N_080W,87,311.744601,"[gross_emissions__AGC__MgCO2, gross_emissions_..."


In [71]:
mismatches[mismatches['tile_id'] == '70N_120W'].sort_values('pct_diff', ascending=False)

,tile_id,variable,year,tile_sum,flox_sum,abs_diff,pct_diff
0,70N_120W,gross_emissions__AGC__MgCO2,2017,2.230077e+04,16610475.0,1.658817e+07,74383.864978
1,70N_120W,gross_emissions__all_C_pools__all_gases__MgCO2e,2017,3.810222e+04,18417430.0,1.837933e+07,48236.891133
2,70N_120W,gross_emissions__all_C_pools__CO2_only__MgCO2,2017,3.740747e+04,17452712.0,1.741530e+07,46555.681967
3,70N_120W,gross_emissions__AGC__MgCO2,2023,1.593311e+05,69547904.0,6.938857e+07,43549.921424
4,70N_120W,gross_emissions__all_C_pools__all_gases__MgCO2e,2023,1.950542e+05,77340992.0,7.714594e+07,39551.030422
...,...,...,...,...,...,...,...
116,70N_120W,net_flux__all_C_pools__all_gases__MgCO2e,2020,-2.269427e+06,-15747826.0,-1.347840e+07,-593.912008
115,70N_120W,net_flux__all_C_pools__CO2_only__MgCO2,2020,-2.269427e+06,-15747826.0,-1.347840e+07,-593.912008
113,70N_120W,net_flux__BGC__MgCO2,2020,-5.218602e+05,-3729383.0,-3.207523e+06,-614.632632
107,70N_120W,net_flux__BGC__MgCO2,2017,-5.486256e+05,-4980774.5,-4.432149e+06,-807.863951


In [86]:
mismatches[mismatches['tile_id'] == '70N_120W'].sort_values('pct_diff', ascending=False)['variable'].unique()

array(['gross_emissions__AGC__MgCO2',
       'gross_emissions__all_C_pools__all_gases__MgCO2e',
       'gross_emissions__all_C_pools__CO2_only__MgCO2',
       'gross_emissions__BGC__MgCO2', 'gross_emissions__CH4__MgCO2e',
       'gross_emissions__all_C_pools__non_CO2_only__MgCO2e',
       'gross_emissions__N2O__MgCO2e', 'net_flux__AGC__MgCO2',
       'net_flux__all_C_pools__all_gases__MgCO2e',
       'net_flux__all_C_pools__CO2_only__MgCO2', 'net_flux__BGC__MgCO2'],
      dtype=object)

In [101]:
chunk_stats_combined[
    (chunk_stats_combined['tile_id'] == '70N_120W') &
    (chunk_stats_combined['pattern'] == 'gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr') &
    (chunk_stats_combined['years'] == '2017') &
    (chunk_stats_combined['data_type'] != 'no data') 
][['chunk_id', 'mean_value', 'sum_value', 'count_value']].sort_values('count_value', ascending=False)

,chunk_id,mean_value,sum_value,count_value
880607,-111_60_-110_61,NaN,NaN,4084003.0
880606,-112_60_-111_61,NaN,NaN,1579109.0
880872,-112_61_-111_62,NaN,NaN,1264883.0
880604,-114_60_-113_61,NaN,NaN,1191877.0
880873,-111_61_-110_62,NaN,NaN,546251.0
...,...,...,...,...
881730,-111_64_-110_65,2.631591e-06,1.417740,8.0
882041,-112_65_-111_66,1.611317e-06,0.839412,4.0
882960,-113_68_-112_69,1.464834e-06,0.671033,4.0
882659,-117_67_-116_68,8.331240e-07,0.401758,1.0


In [106]:
# 1. Are deadwood/litter emissions differences genuinely near zero in flox (both methods agree)?
comparison[
    (comparison['tile_id'] == '70N_120W') &
    (comparison['variable'].isin(['gross_emissions__AGC__MgCO2', 'gross_emissions__BGC__MgCO2', 'gross_emissions__deadwood_C__MgCO2', 'gross_emissions__litter_C__MgCO2']))
].sort_values('year')

,tile_id,variable,year,tile_sum,flox_sum,abs_diff,pct_diff
55620,70N_120W,gross_emissions__AGC__MgCO2,2016,153838.646349,5.291672e+06,5.137833e+06,3.339754e+03
55629,70N_120W,gross_emissions__BGC__MgCO2,2016,25799.020761,1.040606e+06,1.014807e+06,3.933510e+03
55692,70N_120W,gross_emissions__litter_C__MgCO2,2016,250.700996,2.507010e+02,-7.443130e-06,-2.968927e-06
55683,70N_120W,gross_emissions__deadwood_C__MgCO2,2016,501.401992,5.014020e+02,-1.488626e-05,-2.968927e-06
55684,70N_120W,gross_emissions__deadwood_C__MgCO2,2017,275.580191,2.755802e+02,9.447336e-06,3.428162e-06
55693,70N_120W,gross_emissions__litter_C__MgCO2,2017,137.790095,1.377901e+02,4.723668e-06,3.428162e-06
55630,70N_120W,gross_emissions__BGC__MgCO2,2017,15038.235644,8.418245e+05,8.267863e+05,5.497894e+03
55621,70N_120W,gross_emissions__AGC__MgCO2,2017,22300.769442,1.661048e+07,1.658817e+07,7.438386e+04
55622,70N_120W,gross_emissions__AGC__MgCO2,2018,23698.146387,3.571800e+06,3.548102e+06,1.497207e+04
55631,70N_120W,gross_emissions__BGC__MgCO2,2018,20388.448135,1.072673e+06,1.052284e+06,5.161179e+03


In [105]:
# Confirm the NaN pattern exists for AGC pool specifically
for pool_pattern in ['gross_emissions__AGC__MgCO2_ha_yr',
                     'gross_emissions__BGC__MgCO2_ha_yr',
                     'gross_emissions__deadwood_C__MgCO2_ha_yr',
                     'gross_emissions__litter_C__MgCO2_ha_yr']:
    sub = chunk_stats_gross_raw[
        (chunk_stats_gross_raw['tile_id'] == '70N_120W') &
        (chunk_stats_gross_raw['pattern'] == pool_pattern) &
        (chunk_stats_gross_raw['years'] == '2017')
    ]
    n_nan_sum = sub['sum_value'].isna().sum()
    n_str_sum = (sub['sum_value'] == 'N/A- input layer or no per-pixel array supplied').sum()
    n_total = len(sub)
    print(f"{pool_pattern}: {n_total} chunks, {n_nan_sum} NaN sum_value, {n_str_sum} 'N/A' string sum_value")

gross_emissions__AGC__MgCO2_ha_yr: 100 chunks, 40 NaN sum_value, 0 'N/A' string sum_value
gross_emissions__BGC__MgCO2_ha_yr: 100 chunks, 40 NaN sum_value, 0 'N/A' string sum_value
gross_emissions__deadwood_C__MgCO2_ha_yr: 100 chunks, 0 NaN sum_value, 0 'N/A' string sum_value
gross_emissions__litter_C__MgCO2_ha_yr: 100 chunks, 0 NaN sum_value, 0 'N/A' string sum_value
